In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "8"
# pip install scikit-learn xgboost lightgbm imbalanced-learn joblib
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve()))
from src.utils import load_freq, load_sev
from src.modelado import (
    preparar_datos, dividir_datos, armar_xy, ejecutar_experimentos,
    leer_checkpoints, mejores_por_tarea, cargar_modelo, evaluar_prima_pura,
)

df_freq = load_freq("freMTPL2freq.csv")
df_sev  = load_sev("freMTPL2sev.csv")

df = preparar_datos(df_freq, df_sev)
df_train, df_test = dividir_datos(df)          # split por póliza, estratificado por ClaimNb>0

print(f"Train: {len(df_train):,} | Test: {len(df_test):,}")
print(f"% con siniestro  train: {(df_train.ClaimNb>0).mean():.4f} | test: {(df_test.ClaimNb>0).mean():.4f}")

Train: 542,405 | Test: 135,602
% con siniestro  train: 0.0368 | test: 0.0368


In [2]:
MODO_PRUEBA = False      # True: corrida corta para validar; False: corrida completa

if MODO_PRUEBA:
    CARPETA, N_ITER, CV_FOLDS = "models_prueba", 3, 3
    df_tr = df_train.sample(frac=0.05, random_state=42)   # ~27k pólizas
else:
    CARPETA, N_ITER, CV_FOLDS = "models", 15, 5
    df_tr = df_train

N_JOBS, USAR_GPU, SAMPLING_RATIO = -1, False, 0.2   # ratio minoría/mayoría tras SMOTE/ADASYN

MODELOS    = ["GLM", "XGBoost", "LightGBM"]
BALANCEOS  = ["sin_balanceo", "weights", "smote", "adasyn"]

In [5]:
X_tr_f, y_tr_f = armar_xy(df_tr,   "freq")
X_te_f, y_te_f = armar_xy(df_test, "freq")

ck = ejecutar_experimentos(
    "freq", MODELOS, BALANCEOS, X_tr_f, y_tr_f, X_te_f, y_te_f,
    carpeta=CARPETA, n_iter=N_ITER, cv_folds=CV_FOLDS,
    n_jobs=1, threads_modelo=8,          # 8 núcleos físicos
    usar_gpu=USAR_GPU, sampling_ratio=SAMPLING_RATIO,
)

[1/12] freq|GLM|sin_balanceo  -> ya existe en checkpoints, se omite
[2/12] freq|GLM|weights ...
      CV dev=0.25398 | test dev=0.25325 | MAE=0.09521 | RMSE=0.20593 | Gini=0.271 | sesgo=1.649 | 132.4s
[3/12] freq|GLM|smote ...


c:\Users\User\miniconda3\envs\brent-dl\lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


      CV dev=0.37973 | test dev=0.37827 | MAE=0.18219 | RMSE=0.25995 | Gini=0.259 | sesgo=4.174 | 211.2s
[4/12] freq|GLM|adasyn ...
      CV dev=0.39558 | test dev=0.39376 | MAE=0.19121 | RMSE=0.26730 | Gini=0.257 | sesgo=4.430 | 635.0s
[5/12] freq|XGBoost|sin_balanceo ...
      CV dev=0.23619 | test dev=0.23589 | MAE=0.07178 | RMSE=0.20109 | Gini=0.316 | sesgo=0.998 | 303.2s
[6/12] freq|XGBoost|weights ...
      CV dev=0.24271 | test dev=0.24255 | MAE=0.08710 | RMSE=0.20293 | Gini=0.311 | sesgo=1.449 | 292.6s
[7/12] freq|XGBoost|smote ...
      CV dev=0.24943 | test dev=0.24976 | MAE=0.09297 | RMSE=0.20456 | Gini=0.284 | sesgo=1.582 | 510.9s
[8/12] freq|XGBoost|adasyn ...
      CV dev=0.24826 | test dev=0.24809 | MAE=0.09117 | RMSE=0.20413 | Gini=0.289 | sesgo=1.530 | 904.0s
[9/12] freq|LightGBM|sin_balanceo ...
      CV dev=0.23600 | test dev=0.23568 | MAE=0.07184 | RMSE=0.20104 | Gini=0.317 | sesgo=0.998 | 170.2s
[10/12] freq|LightGBM|weights ...
      CV dev=0.24348 | test dev=0.24

In [6]:
X_tr_s, y_tr_s = armar_xy(df_tr,   "sev")
X_te_s, y_te_s = armar_xy(df_test, "sev")

ck = ejecutar_experimentos(
    "sev", MODELOS, ["sin_balanceo"], X_tr_s, y_tr_s, X_te_s, y_te_s,
    carpeta=CARPETA, n_iter=N_ITER, cv_folds=CV_FOLDS,
    n_jobs=N_JOBS, usar_gpu=USAR_GPU,
)

[1/3] sev|GLM|sin_balanceo ...
      CV dev=1.68435 | test dev=1.60403 | MAE=2037.44857 | RMSE=9014.67249 | Gini=0.024 | sesgo=1.032 | 7.4s
[2/3] sev|XGBoost|sin_balanceo ...
      CV dev=1.62261 | test dev=1.54581 | MAE=1883.71547 | RMSE=9002.47457 | Gini=0.109 | sesgo=0.939 | 6.4s
[3/3] sev|LightGBM|sin_balanceo ...
      CV dev=1.66299 | test dev=1.58260 | MAE=1847.53470 | RMSE=9008.62259 | Gini=0.065 | sesgo=0.913 | 4.2s


In [7]:
ck = leer_checkpoints(CARPETA)
cols = ["task","modelo","balanceo","cv_deviance","test_deviance","test_mae","test_rmse",
        "test_gini","test_bias","train_deviance","tiempo_busqueda_s","status"]
display(ck[cols].sort_values(["task","cv_deviance"]).round(5))

display(mejores_por_tarea(ck)[["task","modelo","balanceo","cv_deviance","test_deviance","best_params"]])

,task,modelo,balanceo,cv_deviance,test_deviance,test_mae,test_rmse,test_gini,test_bias,train_deviance,tiempo_busqueda_s,status
8,freq,LightGBM,sin_balanceo,0.23600,0.23568,0.07184,0.20104,0.31708,0.99825,0.23112,170.2,ok
4,freq,XGBoost,sin_balanceo,0.23619,0.23589,0.07178,0.20109,0.31566,0.99844,0.23089,303.2,ok
11,freq,LightGBM,adasyn,0.23976,0.23948,0.07969,0.20227,0.29995,1.22044,0.22879,679.4,ok
10,freq,LightGBM,smote,0.24002,0.23969,0.08012,0.20228,0.29799,1.23214,0.22925,294.8,ok
0,freq,GLM,sin_balanceo,0.24230,0.24168,0.07293,0.20270,0.27056,0.99734,0.24210,88.6,ok
5,freq,XGBoost,weights,0.24271,0.24255,0.08710,0.20293,0.31054,1.44924,0.24047,292.6,ok
9,freq,LightGBM,weights,0.24348,0.24323,0.08797,0.20349,0.31278,1.48534,0.22956,177.6,ok
7,freq,XGBoost,adasyn,0.24826,0.24809,0.09117,0.20413,0.28942,1.52999,0.24619,904.0,ok
6,freq,XGBoost,smote,0.24943,0.24976,0.09297,0.20456,0.28393,1.58217,0.24795,510.9,ok
1,freq,GLM,weights,0.25398,0.25325,0.09521,0.20593,0.27085,1.64880,0.25377,132.4,ok


,task,modelo,balanceo,cv_deviance,test_deviance,best_params
0,freq,LightGBM,sin_balanceo,0.235998,0.235682,"{""estimator__colsample_bytree"": 0.916597455868..."
1,sev,XGBoost,sin_balanceo,1.622605,1.545807,"{""estimator__colsample_bytree"": 0.544246251025..."


In [8]:
mej = mejores_por_tarea(ck).set_index("task")
m_f = cargar_modelo("freq", mej.loc["freq","modelo"], mej.loc["freq","balanceo"], CARPETA)
m_s = cargar_modelo("sev",  mej.loc["sev","modelo"],  mej.loc["sev","balanceo"],  CARPETA)
print(evaluar_prima_pura(m_f, m_s, df_test))

{'mae': 147.42479546558675, 'rmse': 2030.8767650882844, 'tweedie_dev': 65.53359399517993, 'gini': 0.2791744270755343, 'bias': 0.9455859974644072}
